# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a guide for loading and exploring the FAIR<sup>2</sup> dataset using the `mlcroissant` library. The exploration covers loading metadata, reviewing data structures, extracting tabular data, basic processing, and visualization, all referencing Croissant `@id`s as required for reproducibility and FAIRness.

### Dataset Source
FAIR<sup>2</sup> dataset Croissant schema: [https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata (as a Dataset object)
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print("\033[1mName:\033[0m", getattr(metadata, 'name', ''))
print("\033[1mDescription:\033[0m", getattr(metadata, 'description', ''))

## 2. Data Overview

Let's review available record sets, their fields, and their `@id`s in this dataset as described in the Croissant schema.

In [ ]:
# Record sets are described by their '@id' in Croissant.
from pprint import pprint

print("\033[1mAvailable Record Sets:\033[0m")

# Retrieve list of RecordSets and show details:
record_sets = list(dataset.record_sets)
if not record_sets:
    raise ValueError("No record sets found in this dataset.")
for rs in record_sets:
    print(f"\nRecord Set Name: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Description: {rs.description if hasattr(rs,'description') else ''}")
    if hasattr(rs, 'fields'):
        fields = rs.fields
        print(f"  Fields ({len(fields)}):")
        for f in fields:
            print(f"    - {getattr(f, 'name', '')} (@id: {f.id}, dataType: {getattr(f, 'data_type', '-')})")

## 3. Data Extraction
Let's load data from the tabular record set(s) into a pandas DataFrame. We will use the `@id` fields for referencing the specific record set and fields.

Below, we extract all available record sets into DataFrames for analysis. Please note the use of `@id` variables throughout.

In [ ]:
# Prepare to extract all record sets (@id) into DataFrames
dataframes = {}
record_set_ids = [rs.id for rs in record_sets]
print("\033[1mLoading records for record sets:\033[0m", record_set_ids)
for record_set_id in record_set_ids:
    recs = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(recs)
    print(f"\nLoaded DataFrame for Record Set: {record_set_id} (Rows: {dataframes[record_set_id].shape[0]})")
    print("Columns:", dataframes[record_set_id].columns.tolist())

# For demo/exploration, just display the first DataFrame (main clinical data) if available
main_rs_id = record_set_ids[0]  # Use first record set as example
dataframes[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)

We'll apply some typical data processing steps such as filtering, normalization, and grouping on selected fields. All fields are referenced by their Croissant `@id`s for full reproducibility.

First, let's print the available fields of the main record set to choose relevant fields for EDA.

In [ ]:
# Show columns and pick numeric/categorical field candidates
print("Available columns:", dataframes[main_rs_id].columns.tolist())

# Try to identify a numeric field (e.g., 'Age' or similar clinical continuous variable)
numeric_candidates = [col for col in dataframes[main_rs_id].columns if dataframes[main_rs_id][col].dtype in ('int64', 'float64')]  # Guessing by dtype
print("Possible numeric fields:", numeric_candidates)

# If 'Age' is present, use it for demonstration; else pick first available numeric
if 'Age' in dataframes[main_rs_id].columns:
    numeric_field_id = 'Age'  # This is an example; replace with actual Croissant field @id if schema demands.
elif numeric_candidates:
    numeric_field_id = numeric_candidates[0]
else:
    raise ValueError('No numeric field found in the main record set.')

print(f"\nUsing '{numeric_field_id}' as numeric field for EDA.")

# Similarly, look for a group/categorical field
group_candidates = [col for col in dataframes[main_rs_id].columns if dataframes[main_rs_id][col].dtype == 'object']
group_field_id = None
for try_col in ('Sex', 'Gender', 'Location', 'anatomical_location', 'Site'):
    if try_col in dataframes[main_rs_id].columns:
        group_field_id = try_col
        break
if not group_field_id and group_candidates:
    group_field_id = group_candidates[0]  # Arbitrarily pick first categorial/nominal for demonstration
if not group_field_id:
    raise ValueError('No group field found.')

print(f"\nUsing '{group_field_id}' as group field for EDA.")

In [ ]:
# Example: Filtering, normalization, grouping
df = dataframes[main_rs_id].copy()

# Remove NaN or clearly bad values first
df_clean = df[df[numeric_field_id].notnull()]  # Keep only rows with valid numeric field

# Example threshold, e.g. Age > 40 (adapt if different variable)
threshold = 40
filtered_df = df_clean[df_clean[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
display(filtered_df[[group_field_id, numeric_field_id]].head())

# Normalization (z-score)
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"\nNormalized '{numeric_field_id}' (z-score) for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Grouping by group field (categorical): mean for numeric field
if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nGroup-wise mean of '{numeric_field_id}' by '{group_field_id}':")
    display(grouped_df.head())

## 5. Visualization
Let's plot distributions and relationships using the processed DataFrame.

- Distribution of the selected numeric field
- Grouped boxplot by the selected grouping field

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of the numeric field
plt.figure(figsize=(7, 4))
sns.histplot(df_clean[numeric_field_id], bins=15, kde=True, color='royalblue')
plt.title(f'Distribution of {numeric_field_id}')
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.tight_layout()
plt.show()

# Boxplot by group_field (if unique values reasonable)
plt.figure(figsize=(8, 5))
sns.boxplot(data=df_clean, x=group_field_id, y=numeric_field_id, palette='Set2')
plt.title(f'{numeric_field_id} by {group_field_id}')
plt.xlabel(group_field_id)
plt.ylabel(numeric_field_id)
plt.tight_layout()
plt.show()

## 6. Conclusion

In this notebook, we have:
- Loaded a clinical multi-field dataset using the FAIR<sup>2</sup> Croissant schema with `mlcroissant`
- Explored available record sets, fields/columns (referenced throughout by their Croissant `@id`s)
- Converted record sets to pandas DataFrames for analysis
- Performed typical data processing (filtering, normalization, grouping)
- Created visualizations of key clinical features

**This workflow demonstrates how to use the Croissant schema and the `mlcroissant` library for robust, reproducible, and FAIR clinical dataset exploration.**